## Imports

In [ ]:
import jax.numpy as jnp
import jax
import numpy as np
jax.config.update("jax_enable_x64", True)
from src.helpers.initialization import random_clements_init, close_to_identity_clements_init
from src.helpers.initialization import random_haar_init, close_to_identity_haar_init
from src.helpers.initialization import random_butterfly_init, close_to_identity_butterfly_init
from src.helpers.initialization import random_mzi3_init, close_to_identity_mzi3_init
from src.models.mmd_estimator import MMD_loss, MMD_loss_haar, MMD_loss_mzi3, MMD_loss_butterfly
from src.models.training import Trainer
from src.helpers.utils import generate_init_state, pack_params, unpack_params
import matplotlib.pyplot as plt
import seaborn as sns

## Define hyperparameters, generate keys, get dataset

In [ ]:
# Define model hyperparameters
m=100
n=10
optimizer = "Adam"
stepsize = 0.01
n_iters = 2000
n_samples_operators = 2000
n_samples_gurvits = 5000

In [ ]:
# Choose ansatz
# Options: clements, mzi3, butterfly, haar
ansatz = 'haar'

In [ ]:
# Define initial state
# Options: beginning, end, middle_compact, middle_alternating, beginning_alternating
init_state = generate_init_state(m = m, n = n, init_state_type = 'middle_alternating')
init_state_ind = jnp.where(init_state)[0]

In [ ]:
# Get random keys for algorithm
key = jax.random.PRNGKey(0)
key2, init_key = jax.random.split(key, 2)
key3, mmd_key = jax.random.split(key2, 2)
key4, test_key = jax.random.split(key3, 2)
key5, cov_key = jax.random.split(key4, 2)
key6, kgel_key = jax.random.split(key5, 2)

In [ ]:
# Load dataset

# Preference
X_train =  jnp.array(np.loadtxt('src/data/preference_ranking/sushi_train.csv', delimiter = ','))
X_test =  jnp.array(np.loadtxt('src/data/preference_ranking/sushi_test.csv', delimiter = ','))


In [ ]:
# Pick sigma
sigma = 3.0

## Initialize and train model

In [ ]:
# If loading init parameters from another training
#warm_start_path = ''
#initialization_strategy = np.load(warm_start_path)

In [ ]:
# Choose init
perturbation = 0.5
initialization_strategy = "close_to_identity"

In [ ]:
# Get initialization
if isinstance(initialization_strategy, str):
    if initialization_strategy == 'random':
        if ansatz == 'clements':
            U_init, params_mzi_init, gammas_init = random_clements_init(m, init_key)
            params_init = pack_params(params_mzi_init, gammas_init)
        elif ansatz == 'mzi3':
            U_init, params_mzi_init, gammas_init = random_mzi3_init(m, init_key)
            params_init = pack_params(params_mzi_init, gammas_init)
        elif ansatz == 'butterfly':
            U_init, params_mzi_init, gammas_init = random_butterfly_init(m, init_key)
            params_init = pack_params(params_mzi_init, gammas_init)
        elif ansatz == 'haar':
            U_init, params_init = random_haar_init(m, init_key)
    elif initialization_strategy == 'close_to_identity':
        if ansatz == 'clements':
            U_init, params_mzi_init, gammas_init = close_to_identity_clements_init(m,
                                                                                   init_key,
                                                                                   max_value_theta = perturbation,
                                                                                   max_value_phi = perturbation,
                                                                                   max_value_gamma = perturbation)
            params_init = pack_params(params_mzi_init, gammas_init)
        elif ansatz == 'mzi3':
            U_init, params_mzi_init, gammas_init = close_to_identity_mzi3_init(m,
                                                                               init_key,
                                                                               max_value_theta = perturbation,
                                                                               max_value_phi = perturbation,
                                                                               max_value_gamma = perturbation)
            params_init = pack_params(params_mzi_init, gammas_init)
        elif ansatz == 'butterfly':
            U_init, params_mzi_init, gammas_init = close_to_identity_butterfly_init(m,
                                                                                    init_key,
                                                                                    max_value_theta = perturbation,
                                                                                    max_value_phi = perturbation,
                                                                                    max_value_gamma = perturbation)
            params_init = pack_params(params_mzi_init, gammas_init)
        elif ansatz == 'haar':
            U_init, params_init = close_to_identity_haar_init(m, init_key, max_value_perturb = perturbation)
        
elif isinstance(initialization_strategy, np.ndarray): 
    params_init = jnp.array(initialization_strategy)
else:
    print('Initialization not available')

In [ ]:
# Loss function arguments
loss_kwargs = {
    "params": params_init,
    "target_dataset": X_train,
    "sigma": sigma,
    "m": m,
    "n": n,
    "key": mmd_key,
    "n_samples_operators": n_samples_operators,
    "n_samples_gurvits": n_samples_gurvits,
    "init_state_ind": init_state_ind
}

In [ ]:
# Get loss function according to ansatz
if ansatz == "clements":
    Loss = MMD_loss
elif ansatz == "butterfly":
    Loss = MMD_loss_butterfly
elif ansatz == "mzi3":
    Loss = MMD_loss_mzi3
elif ansatz == 'haar':
    Loss = MMD_loss_haar
else:
    print('Ansatz not available')

In [ ]:
# Initialize Trainer class
trainer = Trainer(optimizer = optimizer, loss = Loss, stepsize = stepsize, opt_jit=False)

In [ ]:
# Training
trainer.train(n_iters, loss_kwargs, val_kwargs=None,
              monitor_interval=None, turbo=None,
              convergence_interval=200)

## Analyse results

Get best parameters

In [ ]:
params = jnp.array(trainer.final_params)

In [ ]:
#jnp.save('saved_params', params)

Get loss plot

In [ ]:
plt.plot(trainer.losses, label='train')
plt.xlabel('training step')
plt.ylabel('MMD loss')
#plt.yscale('log')
plt.legend()
plt.show()

In [ ]:
trainer.losses

In [ ]:
final_loss_qcbm = trainer.losses[-1]
final_loss_qcbm

Now get the MMD on test set

In [ ]:
test_loss_qcbm = Loss(circuit_parameters = trainer.final_params,
                 target_dataset = X_test,
                 sigma = sigma,
                 m = m,
                 n = n,
                 key = test_key,
                 n_samples_operators = n_samples_operators,
                 n_samples_gurvits = n_samples_gurvits,
                 init_state_ind = init_state_ind)

In [ ]:
test_loss_qcbm

## Classical benchmarks

Requires [qml_benchmarks](https://github.com/XanaduAI/qml-benchmarks) as a package

In [ ]:
from qml_benchmarks.models.energy_based_model import RestrictedBoltzmannMachine
from qml_benchmarks.model_utils import mmd_loss as mmd_loss_samples
from src.helpers.utils import random_bitstrings
import iqpopt.gen_qml as iqpopt_genqml

Put in right format if one or more bandwidths:

In [ ]:
if jnp.isscalar(sigma):
    sigmas = [sigma]
else:
    sigmas = list(sigma)

### Restricted Boltzmann machine

In [ ]:
n_components_rbm = 64
learning_rate_rbm = 0.01
batch_size_rbm = 64
n_samples_rbm = 1000
n_steps_rbm = 10

In [ ]:
rbm = RestrictedBoltzmannMachine(n_components = n_components_rbm,
                                 learning_rate = learning_rate_rbm,
                                 batch_size = batch_size_rbm,
                                 n_iter = n_iters,
                                 mmd_kwargs={"n_samples": n_samples_rbm, "n_steps": n_steps_rbm, "sigma": sigmas},
                                 score_fn = "mmd",
                                 random_state = np.random.randint(0, 99999))

Train

In [ ]:
rbm.fit(X_train)

Scores on train and test sets

In [ ]:
final_loss_rbm = rbm.score(X_train)
test_loss_rbm = rbm.score(X_test)

In [ ]:
samples = rbm.sample(2000)

In [ ]:
samples.sum(axis = 1)

In [ ]:
-final_loss_rbm

In [ ]:
-test_loss_rbm

### Random model

Generate

In [ ]:
bitstrings = random_bitstrings(m = m, n = n, num_samples = 2000)

Score train and test sets

In [ ]:
random_benchmark_train_avg = []
for sigma in sigmas:
    random_benchmark_train = mmd_loss_samples(ground_truth = X_train,
                                              model_samples = bitstrings,
                                              sigma = sigma)
    random_benchmark_train_avg.append(random_benchmark_train)

sum(random_benchmark_train_avg)/len(random_benchmark_train_avg)

In [ ]:
random_benchmark_test_avg = []
for sigma in sigmas:
    random_benchmark_test = mmd_loss_samples(ground_truth = X_test,
                                              model_samples = bitstrings,
                                              sigma = sigma)
    random_benchmark_test_avg.append(random_benchmark_test)

sum(random_benchmark_test_avg)/len(random_benchmark_test_avg)